In [1]:
import pandas as pd
import tensorflow as tf
import tensorflow_privacy as tfp
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate
from tensorflow.keras.models import Model
import numpy as np
import time
import gc

# 1. Загрузка данных
print("Загружаем данные...")
train_path = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\train.csv'
test_path  = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\test.csv'
valid_path = r'C:\Users\slava\OneDrive\Рабочий стол\PyDissertation\Criteo_x1\valid.csv'

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)
df_valid = pd.read_csv(valid_path)

# 2. Сэмплирование (300 000 строк)
print("Делаем выборку...")
SAMPLE_SIZE = 300000
df_train_sample = df_train.sample(n=SAMPLE_SIZE, random_state=42)
df_test_sample = df_test.sample(n=int(SAMPLE_SIZE * 0.2), random_state=42)
df_valid_sample = df_valid.sample(n=int(SAMPLE_SIZE * 0.2), random_state=42)

# 3. Определение колонок
target = 'label'
dense_features = [f'I{i}' for i in range(1, 14)] 
sparse_features = [f'C{i}' for i in range(1, 27)] 

# 4. Упаковка данных в словари (БЕЗОПАСНЫЙ ФОРМАТ)
def get_keras_data_safe(df, dense_cols, sparse_cols):
    X = {}
    X['dense_inputs'] = df[dense_cols].values.astype(np.float32)
    for col in sparse_cols:
        X[col] = df[col].values.astype(np.int32)
    return X

print("Упаковываем данные для Keras...")
X_train = get_keras_data_safe(df_train_sample, dense_features, sparse_features)
y_train = df_train_sample[target].values.astype(np.float32)

X_valid = get_keras_data_safe(df_valid_sample, dense_features, sparse_features)
y_valid = df_valid_sample[target].values.astype(np.float32)

X_test = get_keras_data_safe(df_test_sample, dense_features, sparse_features)
y_test = df_test_sample[target].values.astype(np.float32)

# Очищаем исходные огромные датафреймы, чтобы освободить RAM перед тяжелым циклом
del df_train, df_test, df_valid, df_train_sample, df_test_sample, df_valid_sample
gc.collect()

print("✅ ДАННЫЕ УСПЕШНО ПОДГОТОВЛЕНЫ И ЗАГРУЖЕНЫ В ПАМЯТЬ!")





Загружаем данные...
Делаем выборку...
Упаковываем данные для Keras...
✅ ДАННЫЕ УСПЕШНО ПОДГОТОВЛЕНЫ И ЗАГРУЖЕНЫ В ПАМЯТЬ!


In [ ]:
# Настройки эксперимента
embedding_sizes = [2, 4, 8]
results_data = []

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', patience=1, restore_best_weights=True, mode='max'
)

def build_clean_model(emb_dim):
    dense_inputs = Input(shape=(len(dense_features),), name='dense_inputs')
    sparse_inputs = {}
    sparse_embeddings = []

    for feat in sparse_features:
        vocab_size = int(max(X_train[feat].max(), X_valid[feat].max(), X_test[feat].max()) + 1)
        inp = Input(shape=(1,), name=feat)
        sparse_inputs[feat] = inp
        emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)(inp)
        sparse_embeddings.append(Flatten()(emb))

    wide_output = dense_inputs
    deep_concat = Concatenate()(sparse_embeddings + [dense_inputs])
    deep_layer = Dense(128, activation='relu')(deep_concat)
    deep_layer = Dense(64, activation='relu')(deep_layer)
    deep_layer = Dense(32, activation='relu')(deep_layer)

    final_concat = Concatenate()([wide_output, deep_layer])
    output = Dense(1, activation='sigmoid', name='output')(final_concat)

    model_inputs = [dense_inputs] + list(sparse_inputs.values())
    return Model(inputs=model_inputs, outputs=output)

print("🚀 Запуск автоматизированного исследования Эмбеддингов (Ablation Study)...\n")
start_total_time = time.time()

for d in embedding_sizes:
    print(f"{'='*50}")
    print(f"ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = {d}")
    print(f"{'='*50}\n")
    
    # --- БАЗОВАЯ МОДЕЛЬ ---
    print("-> 1. Обучение Базовой модели (Baseline)...")
    base_model = build_clean_model(d)
    base_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    base_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                   batch_size=1024, epochs=3, callbacks=[early_stop], verbose=0) 
    
    _, base_auc = base_model.evaluate(X_test, y_test, batch_size=1024, verbose=0)
    print(f"   Базовый AUC: {base_auc:.4f}")
    total_params = base_model.count_params()

    # Сохраняем веса базовой модели
    base_model.save_weights(f'baseline_model_dim{d}.h5')
    print(f"   [Сохранено] baseline_model_dim{d}.h5")
    
    # --- ПРИВАТНАЯ МОДЕЛЬ ---
    print("-> 2. Обучение Приватной модели (DP-SGD)...")
    dp_model = build_clean_model(d)
    dp_optimizer = tfp.DPKerasAdamOptimizer(
        l2_norm_clip=1.0, noise_multiplier=0.1, num_microbatches=1, learning_rate=0.001
    )
    loss = tf.keras.losses.BinaryCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    dp_model.compile(optimizer=dp_optimizer, loss=loss, metrics=['AUC'])
    
    dp_model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                 batch_size=1024, epochs=3, verbose=0)
    
    _, dp_auc = dp_model.evaluate(X_test, y_test, batch_size=1024, verbose=0)
    print(f"   Приватный AUC: {dp_auc:.4f}")
    print(f"   Падение: {(base_auc - dp_auc)*100:.2f}%\n")

    # !!! НОВОЕ !!! Сохраняем веса приватной модели
    dp_model.save_weights(f'dp_model_dim{d}.h5')
    print(f"   [Сохранено] dp_model_dim{d}.h5\n")
    
    # Запись результатов в список
    results_data.append({
        'Размер Эмбеддинга (d)': d,
        'Кол-во параметров': f"{total_params / 1e6:.1f} млн",
        'Базовый AUC': round(base_auc, 4),
        'Приватный AUC (DP)': round(dp_auc, 4),
        'Падение точности': f"{(base_auc - dp_auc):.4f}",
        'В процентах': f"{(base_auc - dp_auc)/base_auc * 100:.2f}%"
    })
    
    del base_model
    del dp_model
    tf.keras.backend.clear_session()
    gc.collect()

print(f"\n✅ Эксперимент завершен! Общее время: {(time.time() - start_total_time)/60:.1f} минут.\n")

# Сохраняем итоговую таблицу в CSV
df_results = pd.DataFrame(results_data)
df_results.to_csv('ablation_results_embeddings.csv', index=False)
print("💾 Таблица с результатами сохранена в файл 'ablation_results_embeddings.csv'")

display(df_results)

🚀 Запуск автоматизированного исследования Эмбеддингов (Ablation Study)...

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 2

-> 1. Обучение Базовой модели (Baseline)...


   Базовый AUC: 0.7669
   [Сохранено] baseline_model_dim2.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6431
   Падение: 12.38%

   [Сохранено] dp_model_dim2.h5

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 4

-> 1. Обучение Базовой модели (Baseline)...
   Базовый AUC: 0.7673
   [Сохранено] baseline_model_dim4.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6676
   Падение: 9.98%

   [Сохранено] dp_model_dim4.h5

ТЕСТИРОВАНИЕ РАЗМЕРНОСТИ EMBEDDING = 8

-> 1. Обучение Базовой модели (Baseline)...
   Базовый AUC: 0.7666
   [Сохранено] baseline_model_dim8.h5
-> 2. Обучение Приватной модели (DP-SGD)...
   Приватный AUC: 0.6655
   Падение: 10.11%

   [Сохранено] dp_model_dim8.h5


✅ Эксперимент завершен! Общее время: 77.8 минут.

💾 Таблица с результатами сохранена в файл 'ablation_results_embeddings.c

,Размер Эмбеддинга (d),Кол-во параметров,Базовый AUC,Приватный AUC (DP),Падение точности,В процентах
0,2,60.3 млн,0.7669,0.6431,0.1238,16.14%
1,4,120.5 млн,0.7673,0.6676,0.0998,13.00%
2,8,241.0 млн,0.7666,0.6655,0.1011,13.19%
